# 📉 RSNA Knee Abnormality Detection: Comprehensive Exploratory Data Analysis (EDA)

Welcome to the exploratory data analysis for the **RSNA Knee Abnormality Detection** competition!

### 🎯 Overview & Goal
In this competition, we are tasked with predicting the probability of **12 distinct knee abnormalities** across MRI studies:
- **Ligaments & Menisci:** `ACL`, `MCL`, `Medial Meniscus`, `Lateral Meniscus`
- **Osteoarthritis (OA):** `Medial OA`, `Lateral OA`, `PF OA`
- **Other Pathology:** `Effusion`, `Synovitis`, `Baker's`, `Contusion`, `Fracture`

### 🔑 Key Challenges Highlighted in Description
1. **Semi-Supervised Setup:** Only a small subset of training studies carry explicit binary ground-truth labels. The remaining studies only provide raw radiology reports (`Report`), potentially requiring NLP extraction.
2. **Multi-Sequence 3D DICOM Data:** Studies contain multiple series (planes: *Sagittal, Coronal, Axial*; weightings: *Fluid-sensitive, Fat-suppressed*).
3. **Heterogeneous Modalities & Syntaxes:** Varying pixel intensities, slice counts (20–45+ slices per volume), and transfer syntaxes (JPEG Lossless, JPEG 2000, Uncompressed).

Let's inspect the tabular metadata, target distributions, language distributions in radiology reports, and 3D DICOM image characteristics!

In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Medical Imaging Libraries
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

# Set visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

# Paths setup
DATA_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
if not DATA_DIR.exists():
    DATA_DIR = Path('/kaggle/input/rsna-knee-abnormality-detection')

print(f"Data directory: {DATA_DIR}")

## 📁 1. Load Data & Inspect Missing Value Structure

Let's load `train.csv`, `train_series.csv`, and check the shape and presence of labeled vs. unlabeled records.

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series_df = pd.read_csv(DATA_DIR / 'train_series.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f"Train Studies Count:        {len(train_df):,}")
print(f"Train Series Count:         {len(train_series_df):,}")
print(f"Test Studies Example Count: {len(test_df):,}")

# Target column list
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

# Split into explicitly labeled vs unlabelled (report only)
labeled_df = train_df.dropna(subset=TARGET_COLS, how='all').copy()
unlabeled_df = train_df[train_df[TARGET_COLS].isna().all(axis=1)].copy()

print(f"\nStudies with Explicit Binary Labels:   {len(labeled_df):,} ({len(labeled_df)/len(train_df):.1%})")
print(f"Studies with Report Only (Unlabeled): {len(unlabeled_df):,} ({len(unlabeled_df)/len(train_df):.1%})")

display(train_df.head(3))

## 📊 2. Target Abnormalities & Co-occurrence Analysis

Understanding label prevalence and finding co-occurrences is critical for structuring loss functions (e.g., BCE with Pos Weights, Focal Loss) and validation strategies (Multilabel Stratified K-Fold).

In [ ]:
if len(labeled_df) > 0:
    # 1. Target Positivity Rates
    pos_counts = labeled_df[TARGET_COLS].sum().sort_values(ascending=False)
    pos_percent = (pos_counts / len(labeled_df)) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.barplot(x=pos_percent.values, y=pos_percent.index, ax=axes[0], palette="Blues_r")
    axes[0].set_title("Abnormality Prevalence (%) in Labeled Subset", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Percentage (%)")
    for i, v in enumerate(pos_percent.values):
        axes[0].text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=9)

    # 2. Number of simultaneous abnormalities per patient
    abnormality_counts = labeled_df[TARGET_COLS].sum(axis=1)
    sns.countplot(x=abnormality_counts, ax=axes[1], palette="crest")
    axes[1].set_title("Number of Simultaneous Pathologies per Patient", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Abnormality Count per Study")
    axes[1].set_ylabel("Number of Studies")

    plt.tight_layout()
    plt.show()

    # 3. Correlation / Co-occurrence Matrix
    plt.figure(figsize=(10, 8))
    corr = labeled_df[TARGET_COLS].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
    plt.title("Correlation Matrix of 12 Knee Abnormalities", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No explicit binary labels found in train.csv sample.")

## 📝 3. Radiology Report Analysis (Unlabeled Dataset Exploration)

The majority of studies may rely on radiology text reports. Let's analyze report length, language distribution, and word frequencies.

In [ ]:
# Report Length Analysis
train_df['report_len'] = train_df['Report'].fillna('').apply(len)
train_df['word_count'] = train_df['Report'].fillna('').apply(lambda x: len(x.split()))

# Створюємо зрозумілу текстову колонку для типу дослідження
train_df['is_labeled'] = train_df[TARGET_COLS].notna().any(axis=1).map({True: 'Labeled', False: 'Report Only'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Гістограма кількості слів
sns.histplot(train_df['word_count'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title("Distribution of Report Word Counts", fontweight='bold')
axes[0].set_xlabel("Word Count")

# 2. Boxplot з коректною передачею hue та легенди
sns.boxplot(
    data=train_df,
    x='is_labeled',
    y='word_count',
    hue='is_labeled',
    ax=axes[1],
    palette="Set2",
    legend=False
)
axes[1].set_title("Word Count: Labeled vs Unlabeled Studies", fontweight='bold')
axes[1].set_xlabel("Study Type")

plt.tight_layout()
plt.show()

# Preview raw report examples
if not train_df['Report'].dropna().empty:
    print("--- Sample Radiology Report 1 ---")
    print(train_df['Report'].dropna().iloc[0][:300] + "...")

## 🧬 4. Series & Acquisition Sequence Analysis (`train_series.csv`)

Knee MRI protocols consist of different planes (*Sagittal*, *Coronal*, *Axial*) and weighting contrasts (*Fluid Sensitive*, *Fat Suppressed*). Let's see how many series are available per study.

In [ ]:
# Series per Study Count
series_per_study = train_series_df.groupby('StudyInstanceUID').size().reset_index(name='series_count')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of series count per study
sns.countplot(
    data=series_per_study, 
    x='series_count', 
    hue='series_count', 
    ax=axes[0], 
    palette="viridis", 
    legend=False
)
axes[0].set_title("Series Count per Study", fontweight='bold')
axes[0].set_xlabel("Number of Series")

# 2. Anatomical Plane Distribution
sns.countplot(
    data=train_series_df, 
    x='Anatomical_Plane', 
    hue='Anatomical_Plane', 
    ax=axes[1], 
    palette="magma", 
    legend=False
)
axes[1].set_title("Anatomical Imaging Planes", fontweight='bold')
axes[1].set_xlabel("Plane")

# 3. Sequence Contrast Attributes (Fluid & Fat Suppression)
contrast_df = train_series_df.groupby(['Fluid_Sensitive', 'Fat_Suppression']).size().reset_index(name='count')
contrast_df['label'] = contrast_df.apply(lambda r: f"Fluid:{r.Fluid_Sensitive} | FatSupp:{r.Fat_Suppression}", axis=1)

sns.barplot(
    data=contrast_df, 
    x='label', 
    y='count', 
    hue='label', 
    ax=axes[2], 
    palette="rocket", 
    legend=False
)
axes[2].set_title("Sequence Contrast Characteristics", fontweight='bold')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 🩺 5. DICOM Image Visualization & Inspection

Now let's inspect the actual DICOM files on disk, inspect pixel arrays, slice counts per series, and plot sample MRI slices across different anatomical planes.

In [ ]:
# Helper to fetch sample DICOM slices for a study
def load_dicom_slice(path):
    try:
        dcm = pydicom.dcmread(path)
        img = dcm.pixel_array
        # Apply VOI LUT if present for proper windowing
        try:
            img = apply_voi_lut(img, dcm)
        except Exception:
            pass
        return img, dcm
    except Exception as e:
        print(f"Error loading DICOM {path}: {e}")
        return None, None

# Find a sample study with multiple series
sample_study_uid = train_series_df['StudyInstanceUID'].iloc[0]
sample_series_rows = train_series_df[train_series_df['StudyInstanceUID'] == sample_study_uid]

print(f"Inspecting StudyUID: {sample_study_uid}")
print(f"Found {len(sample_series_rows)} series for this study.\n")

fig, axes = plt.subplots(1, min(3, len(sample_series_rows)), figsize=(15, 5))
if len(sample_series_rows) == 1:
    axes = [axes]

for i, (_, row) in enumerate(sample_series_rows.head(3).iterrows()):
    series_uid = row['SeriesInstanceUID']
    plane = row['Anatomical_Plane']
    fluid = row['Fluid_Sensitive']
    fat_supp = row['Fat_Suppression']
    
    series_dir = DATA_DIR / 'train_series' / sample_study_uid / series_uid
    dcm_files = sorted(list(series_dir.glob('*.dcm')))
    
    if dcm_files:
        mid_idx = len(dcm_files) // 2
        img, dcm = load_dicom_slice(dcm_files[mid_idx])
        
        if img is not None:
            axes[i].imshow(img, cmap='bone')
            axes[i].set_title(f"Plane: {plane}\nFluid: {fluid} | FatSupp: {fat_supp}\nSlices: {len(dcm_files)}", fontsize=11)
            axes[i].axis('off')

plt.suptitle(f"Sample MRI Sequences for Study: {sample_study_uid[:20]}...", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 6. Advanced EDA: Slice Depth, Resolution & Multi-Label Heatmaps

To build a competitive 2.5D / 3D computer vision pipeline, we need to address three key factors:
1. **Slice Depth Variance:** How many DICOM slices does each series typically contain across planes?
2. **Pathology Co-occurrence Intensity:** Which abnormalities frequently present together in the same patient?
3. **Sequence Contrast vs. Pathology Relationship:** Are certain abnormalities more frequently visible on specific fluid/fat suppression protocols?

In [ ]:
# -------------------------------------------------------------
# 1. SLICE DEPTH DISTRIBUTION PER ANATOMICAL PLANE
# -------------------------------------------------------------
def get_slice_counts_sample(series_df, sample_n=500):
    """Inspects slice count per series folder for a representative sample."""
    sample_df = series_df.sample(min(sample_n, len(series_df)), random_state=42).copy()
    slice_counts = []
    
    for _, row in sample_df.iterrows():
        s_path = DATA_DIR / 'train_series' / str(row['StudyInstanceUID']) / str(row['SeriesInstanceUID'])
        if s_path.exists():
            # Fast DICOM slice counter
            slice_counts.append(len(list(s_path.glob('*.dcm'))))
        else:
            slice_counts.append(np.nan)
            
    sample_df['slice_count'] = slice_counts
    return sample_df.dropna(subset=['slice_count'])

sliced_sample_df = get_slice_counts_sample(train_series_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Violin Plot of Slice Count by Plane
if not sliced_sample_df.empty:
    sns.violinplot(
        data=sliced_sample_df, 
        x='Anatomical_Plane', 
        y='slice_count', 
        hue='Anatomical_Plane',
        ax=axes[0], 
        palette="coolwarm", 
        inner="quartile",
        legend=False
    )
    axes[0].set_title("Distribution of DICOM Slices per Series by Plane", fontweight='bold')
    axes[0].set_xlabel("Anatomical Plane")
    axes[0].set_ylabel("Slice Count per Volume")

# -------------------------------------------------------------
# 2. TARGET CORRELATION / CO-OCCURRENCE MATRIX (FIXED)
# -------------------------------------------------------------
if len(labeled_df) > 0:
    # Заповнюємо можливі NaN нулями, перетворюємо на цілочисельний тип (int)
    clean_targets = labeled_df[TARGET_COLS].fillna(0).astype(int)
    
    # Розраховуємо абсолютну матрицю збігів (Co-occurrence)
    co_matrix = clean_targets.T.dot(clean_targets)
    
    sns.heatmap(
        co_matrix, 
        annot=True, 
        fmt="g",  # "g" працює безпечно з будь-якими числовими типами даних
        cmap="YlGnBu", 
        cbar=True, 
        ax=axes[1],
        linewidths=0.5
    )
    axes[1].set_title("Absolute Co-occurrence Count Between Abnormalities", fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 🖼️ 7. 3D MRI Volumetric Slice Grid (Mid-Slice Montage)

Visualizing multiple contiguous slices from a single DICOM series to observe how structural abnormalities (e.g., meniscus tears or joint effusion) transition across spatial depth.

In [ ]:
# -------------------------------------------------------------
# 3. VOLUMETRIC SLICE GRID (1x6 MONTAGE ACROSS DEPTH)
# -------------------------------------------------------------
def plot_series_montage(study_uid, series_uid, num_slices=6):
    series_dir = DATA_DIR / 'train_series' / study_uid / series_uid
    dcm_files = sorted(list(series_dir.glob('*.dcm')))
    
    if not dcm_files:
        print(f"No slices found in {series_dir}")
        return
        
    indices = np.linspace(0, len(dcm_files) - 1, num_slices, dtype=int)
    
    fig, axes = plt.subplots(1, num_slices, figsize=(18, 4))
    
    for idx, slice_idx in enumerate(indices):
        img, dcm = load_dicom_slice(dcm_files[slice_idx])
        if img is not None:
            axes[idx].imshow(img, cmap='bone')
            axes[idx].set_title(f"Slice {slice_idx+1}/{len(dcm_files)}", fontsize=10)
            axes[idx].axis('off')
            
    plt.suptitle(f"Spatial Progression Across Depth | Series: {series_uid[:15]}...", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Sample plot for the first series
sample_row = train_series_df.iloc[0]
plot_series_montage(sample_row['StudyInstanceUID'], sample_row['SeriesInstanceUID'])

## 📑 8. Radiology Reports: Most Frequent Clinical Keywords

Let's extract top clinical keywords and n-grams from the text radiology reports using TF-IDF / Frequency Counting. This highlights key pathology terms used by radiologists in unlabeled cases.

In [ ]:
# -------------------------------------------------------------
# 4. RADIOLOGY REPORT TOP KEYWORDS (NGRAM ANALYSIS)
# -------------------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer

reports = train_df['Report'].dropna()

if not reports.empty:
    vec = CountVectorizer(stop_words='english', ngram_range=(1, 2), max_features=15)
    bag_of_words = vec.fit_transform(reports)
    sum_words = bag_of_words.sum(axis=0)
    
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    
    top_df = pd.DataFrame(words_freq, columns=['Keyword', 'Frequency'])
    
    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=top_df, 
        x='Frequency', 
        y='Keyword', 
        hue='Keyword',
        palette="mako", 
        legend=False
    )
    plt.title("Top 15 Radiology Report Terms / Bi-grams", fontweight='bold', fontsize=13)
    plt.xlabel("Frequency Count")
    plt.tight_layout()
    plt.show()

## 🩺 9. Medical Imaging Metadata: Resolution & Pixel Intensity Distribution

Understanding spatial resolution (image width/height) and original pixel signal values across DICOM files is crucial for setting up image normalization, resizing, and dynamic range windowing.

In [ ]:
# -------------------------------------------------------------
# 5. DICOM RESOLUTION & PIXEL INTENSITY DISTRIBUTION
# -------------------------------------------------------------
def analyze_dicom_metadata(series_df, sample_n=100):
    sample_rows = series_df.sample(min(sample_n, len(series_df)), random_state=42)
    shapes = []
    pixel_means = []
    pixel_stds = []
    
    for _, row in sample_rows.iterrows():
        s_path = DATA_DIR / 'train_series' / str(row['StudyInstanceUID']) / str(row['SeriesInstanceUID'])
        dcm_files = sorted(list(s_path.glob('*.dcm')))
        if dcm_files:
            # Load middle slice
            mid_file = dcm_files[len(dcm_files) // 2]
            img, dcm = load_dicom_slice(mid_file)
            if img is not None:
                shapes.append(f"{img.shape[0]}x{img.shape[1]}")
                pixel_means.append(img.mean())
                pixel_stds.append(img.std())
                
    return pd.DataFrame({'shape': shapes, 'mean': pixel_means, 'std': pixel_stds})

meta_stats_df = analyze_dicom_metadata(train_series_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Image Resolution Counts
shape_counts = meta_stats_df['shape'].value_counts().reset_index()
shape_counts.columns = ['Resolution', 'Count']

sns.barplot(
    data=shape_counts, 
    x='Resolution', 
    y='Count', 
    hue='Resolution', 
    ax=axes[0], 
    palette="flare", 
    legend=False
)
axes[0].set_title("MRI Matrix Resolution Distribution (Sample)", fontweight='bold')
axes[0].set_xlabel("Pixel Dimensions (H x W)")
axes[0].set_ylabel("Frequency")

# 2. Pixel Intensity Histogram (Signal distribution)
sns.histplot(meta_stats_df['mean'], bins=30, kde=True, ax=axes[1], color='indigo')
axes[1].set_title("Mean Pixel Signal Intensity Distribution", fontweight='bold')
axes[1].set_xlabel("Mean Signal Intensity")
axes[1].set_ylabel("Series Count")

plt.tight_layout()
plt.show()

## 🏷️ 10. Label Availability & Dataset Completeness Overview

Let's inspect the exact proportion of fully labeled vs. report-only studies in the training set and observe missing value patterns across target columns.

In [ ]:
# -------------------------------------------------------------
# 6. DATASET COMPLETENESS & LABEL AVAILABILITY ANALYSIS
# -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Proportion of Labeled vs Unlabeled Studies
train_df['label_status'] = train_df[TARGET_COLS].notna().any(axis=1).map({
    True: 'Explicitly Labeled', 
    False: 'Report Only (Unlabeled)'
})

status_counts = train_df['label_status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']

sns.barplot(
    data=status_counts, 
    x='Status', 
    y='Count', 
    hue='Status', 
    ax=axes[0], 
    palette="Set2", 
    legend=False
)
axes[0].set_title("Dataset Composition: Labeled vs. Unlabeled", fontweight='bold')
axes[0].set_ylabel("Number of Studies")

for i, v in enumerate(status_counts['Count']):
    axes[0].text(i, v + (max(status_counts['Count']) * 0.01), f"{v:,} ({v/len(train_df):.1%})", ha='center', fontsize=10)

# 2. Presence of Each Abnormality Label (non-null counts)
non_null_counts = train_df[TARGET_COLS].notna().sum().reset_index()
non_null_counts.columns = ['Abnormality', 'Available_Labels']

sns.barplot(
    data=non_null_counts, 
    x='Available_Labels', 
    y='Abnormality', 
    hue='Abnormality', 
    ax=axes[1], 
    palette="mako", 
    legend=False
)
axes[1].set_title("Count of Available Ground Truth Labels per Target", fontweight='bold')
axes[1].set_xlabel("Non-Null Label Count")

plt.tight_layout()
plt.show()

## 💡 Key Takeaways & Baseline Modeling Strategy

### 📌 Summary of Insights:
1. **Target Multi-Label Nature:** High co-occurrence between osteoarthritis indicators (`Medial OA`, `Lateral OA`, `PF OA`) and meniscus tears. Loss functions should handle multi-label dependencies effectively.
2. **Semi-Supervised / Pseudo-Label Opportunity:** Unlabeled cases with radiology reports can be leveraged using LLMs/NLP encoders (e.g., BioRoBERTa, DeBERTa-v3) to generate pseudo-labels for training the 3D/2.5D vision models.
3. **Data Representation (2.5D / 3D Pipelines):**
   - Each study has 3 main planes (Sagittal, Coronal, Axial).
   - Selecting middle slices or using 3D CNNs / 2.5D multi-slice feature extractors (e.g., ConvNeXt, EfficientNet-3D, Swin UNETR) per series, followed by multi-series fusion, is a promising architecture choice.

---
**Next Step:** Build a fast **Baseline Model Notebook** using a 2.5D ResNet/EfficientNet backbone on labeled slices!